In [1]:
!pip install mlflow boto3 awscli optuna xgboost imbalanced-learn


   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   ---------------------------------------- 2/2 [optuna]



In [2]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-18-233-223-218.compute-1.amazonaws.com:5000/")

In [3]:
# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning")

2026/03/10 16:12:32 INFO mlflow.tracking.fluent: Experiment with name 'Exp 5 - ML Algos with HP Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://ytmlflow-bucket-2026/5', creation_time=1773139351931, experiment_id='5', last_update_time=1773139351931, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning', tags={}>

In [4]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

c:\Users\shrut\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
df = pd.read_csv('reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [8]:
import os
import pickle
from xgboost import XGBClassifier
import optuna

# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})
df = df.dropna(subset=['category'])

ngram_range = (1, 3)
max_features = 5000  # reduced from 10000 for speed

# Step 2: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_comment'], df['category'], 
    test_size=0.2, random_state=42, stratify=df['category']
)

# Step 3: Vectorization
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Step 4: SMOTE
smote = SMOTE(random_state=42)
X_train_vec, y_train = smote.fit_resample(X_train_vec, y_train)

# Step 5: MLflow logging function
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.log_param("algo_name", model_name)

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Save model as pickle instead of log_model (fixes 404 error)
        import pickle
        model_filename = f"{model_name}_model.pkl"
        with open(model_filename, "wb") as f:
            pickle.dump(model, f)
        mlflow.log_artifact(model_filename)
        os.remove(model_filename)

        print(f"✅ {model_name} | Accuracy: {accuracy:.4f}")

# Step 6: Optuna objective function - optimized for speed
def objective_xgboost(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 200)      # reduced from 300
    learning_rate = trial.suggest_float('learning_rate', 1e-3, 1e-1, log=True)  # tightened range
    max_depth = trial.suggest_int('max_depth', 3, 7)               # reduced from 10

    model = XGBClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        random_state=42,
        n_jobs=-1,          # use all CPU cores
        tree_method='hist'  # faster algorithm
    )
    return accuracy_score(y_test, model.fit(X_train_vec, y_train).predict(X_test_vec))

# Step 7: Run Optuna and log best model
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_xgboost, n_trials=10)  # reduced from 30

    best_params = study.best_params
    print(f"Best params: {best_params}")

    best_model = XGBClassifier(
        n_estimators=best_params['n_estimators'],
        learning_rate=best_params['learning_rate'],
        max_depth=best_params['max_depth'],
        random_state=42,
        n_jobs=-1,
        tree_method='hist'
    )
    log_mlflow("XGBoost", best_model, X_train_vec, X_test_vec, y_train, y_test)

run_optuna_experiment()

[I 2026-03-10 19:34:14,563] A new study created in memory with name: no-name-a47e3af4-a4cb-4c8e-bc1a-653afee7c95e
[I 2026-03-10 19:34:21,721] Trial 0 finished with value: 0.8569417561147281 and parameters: {'n_estimators': 181, 'learning_rate': 0.06160711253456847, 'max_depth': 6}. Best is trial 0 with value: 0.8569417561147281.
[I 2026-03-10 19:34:26,382] Trial 1 finished with value: 0.7355270103818405 and parameters: {'n_estimators': 81, 'learning_rate': 0.009841623219320542, 'max_depth': 6}. Best is trial 0 with value: 0.8569417561147281.
[I 2026-03-10 19:34:29,711] Trial 2 finished with value: 0.6609185289459792 and parameters: {'n_estimators': 102, 'learning_rate': 0.0030145793470272766, 'max_depth': 4}. Best is trial 0 with value: 0.8569417561147281.
[I 2026-03-10 19:34:33,032] Trial 3 finished with value: 0.729016364596164 and parameters: {'n_estimators': 77, 'learning_rate': 0.013462147080922153, 'max_depth': 5}. Best is trial 0 with value: 0.8569417561147281.
[I 2026-03-10 19:

Best params: {'n_estimators': 181, 'learning_rate': 0.06160711253456847, 'max_depth': 6}
✅ XGBoost | Accuracy: 0.8569
🏃 View run XGBoost_SMOTE_TFIDF_Trigrams at: http://ec2-18-233-223-218.compute-1.amazonaws.com:5000/#/experiments/5/runs/631d2f56a5da47f2a9786b6238357622
🧪 View experiment at: http://ec2-18-233-223-218.compute-1.amazonaws.com:5000/#/experiments/5
